# Python asyncio
Covers: async/await, event loop, coroutines, Tasks, gather/wait/create_task, asyncio.Queue, async context managers, async generators

## 1. async/await Basics

In [ ]:
import asyncio
import time

# Basic coroutine
async def greet(name, delay):
    print(f'Hello, {name}!')
    await asyncio.sleep(delay)  # non-blocking sleep
    print(f'Goodbye, {name}!')
    return f'Done with {name}'

# Calling a coroutine returns a coroutine object (doesn't run!)
coro = greet('Alice', 0.1)
print('Type:', type(coro))  # <class 'coroutine'>
print('Not running yet!')

# Run with asyncio.run()
result = asyncio.run(greet('Alice', 0.1))
print('Result:', result)

# Sequential vs concurrent
async def sequential():
    start = time.perf_counter()
    await greet('Alice', 0.2)
    await greet('Bob', 0.2)
    await greet('Charlie', 0.2)
    print(f'Sequential: {time.perf_counter()-start:.2f}s')

async def concurrent():
    start = time.perf_counter()
    await asyncio.gather(
        greet('Alice', 0.2),
        greet('Bob', 0.2),
        greet('Charlie', 0.2),
    )
    print(f'Concurrent: {time.perf_counter()-start:.2f}s')

print('\n--- Sequential ---')
asyncio.run(sequential())
print('\n--- Concurrent ---')
asyncio.run(concurrent())

## 2. asyncio.gather, create_task, wait

In [ ]:
import asyncio
import time
import random

async def fetch(name, delay):
    await asyncio.sleep(delay)
    return f'{name}: {delay:.2f}s'

# asyncio.gather — concurrent, ordered results
async def demo_gather():
    start = time.perf_counter()
    results = await asyncio.gather(
        fetch('A', 0.3),
        fetch('B', 0.1),
        fetch('C', 0.2),
    )
    elapsed = time.perf_counter() - start
    print(f'gather results (input order): {results}')
    print(f'Time: {elapsed:.2f}s (max delay was 0.3s)')

asyncio.run(demo_gather())

# asyncio.create_task — schedule immediately
async def demo_create_task():
    print('\ncreate_task demo:')
    t1 = asyncio.create_task(fetch('Task-A', 0.3), name='task-a')
    t2 = asyncio.create_task(fetch('Task-B', 0.1), name='task-b')
    
    print('Tasks created, doing other work...')
    await asyncio.sleep(0.05)  # other work
    
    r1 = await t1
    r2 = await t2
    print(f'Results: {r1}, {r2}')
    
    # Cancel a task
    t3 = asyncio.create_task(fetch('Task-C', 10))
    await asyncio.sleep(0.05)
    t3.cancel()
    try:
        await t3
    except asyncio.CancelledError:
        print('Task-C was cancelled')

asyncio.run(demo_create_task())

# asyncio.wait — more control
async def demo_wait():
    print('\nwait demo (FIRST_COMPLETED):')
    tasks = [asyncio.create_task(fetch(f'T{i}', random.uniform(0.1, 0.5))) for i in range(5)]
    
    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
    print(f'First done: {len(done)}, still pending: {len(pending)}')
    for t in done:
        print(f'  Result: {t.result()}')
    
    # Cancel remaining
    for t in pending:
        t.cancel()
    await asyncio.gather(*pending, return_exceptions=True)

asyncio.run(demo_wait())

## 3. asyncio.Queue — Producer/Consumer

In [ ]:
import asyncio
import random
import time

async def producer(queue, n, name):
    for i in range(n):
        item = random.randint(1, 100)
        await queue.put(item)
        print(f'{name} produced: {item} (queue size: {queue.qsize()})')
        await asyncio.sleep(random.uniform(0.05, 0.15))

async def consumer(queue, name):
    total = 0
    count = 0
    while True:
        try:
            item = await asyncio.wait_for(queue.get(), timeout=0.5)
            total += item
            count += 1
            print(f'{name} consumed: {item}')
            queue.task_done()
        except asyncio.TimeoutError:
            break  # no more items
    return total, count

async def main():
    queue = asyncio.Queue(maxsize=5)
    
    # Start producers and consumers concurrently
    prod1 = asyncio.create_task(producer(queue, 5, 'Producer-1'))
    prod2 = asyncio.create_task(producer(queue, 5, 'Producer-2'))
    cons1 = asyncio.create_task(consumer(queue, 'Consumer-1'))
    cons2 = asyncio.create_task(consumer(queue, 'Consumer-2'))
    
    await asyncio.gather(prod1, prod2)
    r1 = await cons1
    r2 = await cons2
    
    print(f'\nConsumer-1: {r1[1]} items, sum={r1[0]}')
    print(f'Consumer-2: {r2[1]} items, sum={r2[0]}')
    print(f'Total items: {r1[1] + r2[1]}')

asyncio.run(main())

## 4. Async Context Managers

In [ ]:
import asyncio
from contextlib import asynccontextmanager

# Class-based async context manager
class AsyncConnection:
    def __init__(self, host, port):
        self.host = host
        self.port = port
        self.connected = False
    
    async def __aenter__(self):
        print(f'Connecting to {self.host}:{self.port}...')
        await asyncio.sleep(0.1)  # simulate connection
        self.connected = True
        return self
    
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print(f'Closing connection to {self.host}:{self.port}')
        await asyncio.sleep(0.05)  # simulate cleanup
        self.connected = False
        return False
    
    async def query(self, sql):
        if not self.connected:
            raise RuntimeError('Not connected')
        await asyncio.sleep(0.05)  # simulate query
        return f'Result of: {sql}'

# Generator-based async context manager
@asynccontextmanager
async def managed_session(name):
    print(f'Opening session: {name}')
    await asyncio.sleep(0.05)
    session = {'name': name, 'active': True}
    try:
        yield session
    except Exception as e:
        print(f'Session error: {e}')
        raise
    finally:
        session['active'] = False
        print(f'Closing session: {name}')
        await asyncio.sleep(0.05)

async def main():
    # Class-based
    async with AsyncConnection('localhost', 5432) as conn:
        result = await conn.query('SELECT * FROM users')
        print('Query result:', result)
    
    # Generator-based
    async with managed_session('api-session') as session:
        print('Using session:', session['name'])
        await asyncio.sleep(0.1)
    
    # Multiple async context managers
    async with AsyncConnection('db1', 5432) as db1, \
               AsyncConnection('db2', 5433) as db2:
        r1 = await db1.query('SELECT 1')
        r2 = await db2.query('SELECT 2')
        print('Both queries:', r1, r2)

asyncio.run(main())

## 5. Async Generators

In [ ]:
import asyncio

# Async generator
async def async_range(start, stop, delay=0.05):
    """Async generator that yields values with delays."""
    for i in range(start, stop):
        await asyncio.sleep(delay)
        yield i

# Simulate paginated API
async def paginated_api(total_items, page_size=3):
    """Simulate fetching paginated data."""
    page = 0
    fetched = 0
    while fetched < total_items:
        await asyncio.sleep(0.05)  # simulate API call
        batch = list(range(fetched, min(fetched + page_size, total_items)))
        for item in batch:
            yield item
        fetched += len(batch)
        page += 1

async def main():
    # async for loop
    print('async for loop:')
    async for value in async_range(0, 5):
        print(f'  Got: {value}')
    
    # async list comprehension
    squares = [x**2 async for x in async_range(1, 6, delay=0.02)]
    print('\nAsync comprehension:', squares)
    
    # Paginated API
    print('\nPaginated API (10 items, page_size=3):')
    all_items = []
    async for item in paginated_api(10, page_size=3):
        all_items.append(item)
    print('All items:', all_items)

asyncio.run(main())

## 6. Error Handling and TaskGroup (Python 3.11+)

In [ ]:
import asyncio
import sys

async def might_fail(n):
    await asyncio.sleep(0.05)
    if n % 3 == 0:
        raise ValueError(f'Task {n} failed!')
    return n * 2

# gather with return_exceptions=True
async def handle_errors_gather():
    results = await asyncio.gather(
        *[might_fail(i) for i in range(7)],
        return_exceptions=True
    )
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            print(f'  Task {i}: ERROR — {result}')
        else:
            print(f'  Task {i}: {result}')

print('gather with return_exceptions=True:')
asyncio.run(handle_errors_gather())

# run_in_executor — run blocking code in async context
import time as time_module
from concurrent.futures import ThreadPoolExecutor

def blocking_computation(n):
    """Blocking function that can't be awaited."""
    time_module.sleep(0.1)  # blocking sleep
    return sum(i * i for i in range(n))

async def async_with_blocking():
    loop = asyncio.get_event_loop()
    # Run blocking code in thread pool — doesn't block event loop
    with ThreadPoolExecutor(max_workers=4) as pool:
        tasks = [
            loop.run_in_executor(pool, blocking_computation, 100_000)
            for _ in range(4)
        ]
        results = await asyncio.gather(*tasks)
    print('\nrun_in_executor results:', results[:2], '...')

asyncio.run(async_with_blocking())

# TaskGroup (Python 3.11+)
if sys.version_info >= (3, 11):
    async def use_task_group():
        print('\nTaskGroup (Python 3.11+):')
        async with asyncio.TaskGroup() as tg:
            t1 = tg.create_task(might_fail(1))
            t2 = tg.create_task(might_fail(2))
            t4 = tg.create_task(might_fail(4))
        print(f'Results: {t1.result()}, {t2.result()}, {t4.result()}')
    
    asyncio.run(use_task_group())
else:
    print(f'\nTaskGroup requires Python 3.11+ (you have {sys.version[:6]})')